# 2D Soft Body Dynamics: Explicit Integration

This tutorial implements 2D soft body simulation using **NVIDIA Warp** with explicit time integration.

**Learning Goals:**
1. Understand explicit (forward) Euler time integration
2. Implement spring forces using Hooke's Law
3. **See why explicit integration becomes UNSTABLE with large timesteps**

**Key Demonstration:** We'll show that explicit integration EXPLODES when timesteps are too large!

**Prerequisites:** Basic understanding of physics simulation and Python

## 1. The Physics

Soft bodies follow Newton's second law:
$$\mathbf{M}\ddot{\mathbf{x}} = \mathbf{f}(\mathbf{x}, \mathbf{v})$$

where:
- $\mathbf{M}$ is the mass matrix
- $\mathbf{x}$ is position, $\mathbf{v} = \dot{\mathbf{x}}$ is velocity
- $\mathbf{f}$ includes spring forces, gravity, and damping

## 2. Explicit Euler Integration

Explicit (Forward) Euler computes forces at the **current** time:

$$\mathbf{v}_{n+1} = \mathbf{v}_n + \Delta t \cdot \mathbf{M}^{-1} \mathbf{f}(\mathbf{x}_n)$$
$$\mathbf{x}_{n+1} = \mathbf{x}_n + \Delta t \cdot \mathbf{v}_{n+1}$$

**Pros:**
- Simple to implement
- Fast per step

**Cons:**
- ⚠️ **Conditionally stable**: requires tiny timesteps ($\Delta t < 2\sqrt{m/k}$)
- ⚠️ **EXPLODES** if timestep is too large!

### Stability Limit
For spring stiffness $k$ and mass $m$, explicit Euler is stable only when:
$$\Delta t < \frac{2}{\omega} = 2\sqrt{\frac{m}{k}}$$

Higher stiffness → smaller stability limit. We'll demonstrate this below!

## 3. Spring Force Model (Hooke's Law)

For a spring connecting particles $i$ and $j$:

$$\mathbf{f}_{spring} = k_s (L - L_0) \hat{\mathbf{d}} + k_d \dot{L} \hat{\mathbf{d}}$$

where:
- $k_s$ = spring stiffness
- $k_d$ = damping coefficient
- $L$ = current length, $L_0$ = rest length
- $\hat{\mathbf{d}}$ = unit direction vector

---
## 4. Implementation

Step-by-step GPU implementation using Warp.

In [1]:
# Step 1: Imports
import numpy as np
import warp as wp
from scipy.spatial import Delaunay
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import Image
import time

# Note: All kernels and classes defined in this notebook are also available in utils.py
# for reuse in subsequent tutorials. Import them with:
#   from utils import State, Model, SolverExplicit, run_simulation, get_colors, ...

wp.init()
print(f"Warp {wp.__version__} on {wp.get_device()}")

Warp 1.11.0.dev20251123 initialized:
   Git commit: 8b8f0b85ca54c0026574f834764e26615056aef6
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA L40S" (44 GiB, sm_89, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0.dev20251123
Warp 1.11.0.dev20251123 on cuda:0


### 4.1 Spring Force Kernel

Compute spring forces using Hooke's Law:
- **Spring force:** $f = k_s(L - L_0)$
- **Damping:** $f_d = k_d \dot{L}$

In [2]:
# Step 2: Spring Force Kernel
@wp.kernel
def eval_spring_2d(
    x: wp.array(dtype=wp.vec2),           # Positions
    v: wp.array(dtype=wp.vec2),           # Velocities
    spring_indices: wp.array(dtype=int),   # Particle pairs
    spring_rest_lengths: wp.array(dtype=float),
    spring_stiffness: wp.array(dtype=float),
    spring_damping: wp.array(dtype=float),
    f: wp.array(dtype=wp.vec2),           # Output forces
    spring_strains: wp.array(dtype=float), # Output strains
):
    tid = wp.tid()
    i = spring_indices[tid * 2 + 0]
    j = spring_indices[tid * 2 + 1]
    if i == -1 or j == -1:
        return
    
    ke = spring_stiffness[tid]
    kd = spring_damping[tid]
    rest = spring_rest_lengths[tid]
    
    xij = x[i] - x[j]
    vij = v[i] - v[j]
    L = wp.length(xij)
    
    if L < 1e-6:
        spring_strains[tid] = 0.0
        return
    
    d_hat = xij / L
    extension = L - rest
    L_dot = wp.dot(d_hat, vij)
    
    spring_strains[tid] = extension / rest
    force = d_hat * (ke * extension + kd * L_dot)
    
    wp.atomic_sub(f, i, force)
    wp.atomic_add(f, j, force)

print("Spring kernel defined")

Spring kernel defined


### 4.2 Integration Kernels

**Symplectic Euler** (leapfrog) for better energy conservation:
1. Half-kick velocity
2. Drift position  
3. Second half-kick

In [3]:
# Step 3: Integration Kernels
@wp.kernel
def integrate_particles_2d(
    x: wp.array(dtype=wp.vec2), v: wp.array(dtype=wp.vec2),
    f: wp.array(dtype=wp.vec2), inv_mass: wp.array(dtype=float),
    gravity: wp.vec2, dt: float,
    x_new: wp.array(dtype=wp.vec2), v_new: wp.array(dtype=wp.vec2),
):
    tid = wp.tid()
    acc = f[tid] * inv_mass[tid] + gravity
    v_half = v[tid] + acc * (dt / 2.0)
    x_new[tid] = x[tid] + v_half * dt
    v_new[tid] = v_half


@wp.kernel
def finalize_velocity_2d(
    v: wp.array(dtype=wp.vec2), f: wp.array(dtype=wp.vec2),
    inv_mass: wp.array(dtype=float), gravity: wp.vec2, dt: float,
    v_new: wp.array(dtype=wp.vec2),
):
    tid = wp.tid()
    acc = f[tid] * inv_mass[tid] + gravity
    v_new[tid] = v[tid] + acc * (dt / 2.0)


@wp.kernel
def apply_boundary_2d(x: wp.array(dtype=wp.vec2), v: wp.array(dtype=wp.vec2), boxsize: float):
    tid = wp.tid()
    pos = x[tid]
    vel = v[tid]
    if pos[0] < 0.0:
        pos = wp.vec2(-pos[0], pos[1])
        vel = wp.vec2(-vel[0], vel[1])
    elif pos[0] > boxsize:
        pos = wp.vec2(2.0*boxsize - pos[0], pos[1])
        vel = wp.vec2(-vel[0], vel[1])
    if pos[1] < 0.0:
        pos = wp.vec2(pos[0], -pos[1])
        vel = wp.vec2(vel[0], -vel[1])
    elif pos[1] > boxsize:
        pos = wp.vec2(pos[0], 2.0*boxsize - pos[1])
        vel = wp.vec2(vel[0], -vel[1])
    x[tid] = pos
    v[tid] = vel

print("Integration kernels defined")

Integration kernels defined


### 4.3 State and Model Classes

Data structures for particle positions, velocities, and spring connectivity.

In [4]:
# Step 4: State and Model Classes
class State:
    def __init__(self):
        self.particle_q = None
        self.particle_qd = None
        self.particle_f = None


class Model:
    def __init__(self, device='cuda'):
        self.device = wp.get_device(device)
        self.particle_q = None
        self.particle_qd = None
        self.particle_mass = None
        self.particle_inv_mass = None
        self.particle_count = 0
        self.spring_indices = None
        self.spring_rest_length = None
        self.spring_stiffness = None
        self.spring_damping = None
        self.spring_count = 0
        self.spring_strains = None
        self.gravity = None
        self.boxsize = 3.0
    
    def state(self):
        s = State()
        if self.particle_count > 0:
            s.particle_q = wp.clone(self.particle_q)
            s.particle_qd = wp.clone(self.particle_qd)
            s.particle_f = wp.zeros(self.particle_count, dtype=wp.vec2, device=self.device)
        return s
    
    def set_gravity(self, g):
        if self.gravity is None:
            self.gravity = wp.zeros(1, dtype=wp.vec2, device=self.device)
        self.gravity.assign([wp.vec2(g[0], g[1])])
    
    @classmethod
    def from_circle(cls, radius=0.5, num_boundary=20, num_rings=3, device='cuda', boxsize=3.0, center=None):
        model = cls(device=device)
        model.boxsize = boxsize
        if center is None:
            center = (boxsize/2.0, boxsize/2.0)
        cx, cy = center
        
        all_pts = []
        angles = np.linspace(0, 2*np.pi, num_boundary, endpoint=False)
        all_pts.append(np.c_[radius*np.cos(angles), radius*np.sin(angles)])
        for ring in range(1, num_rings+1):
            r = radius * (num_rings-ring+1) / (num_rings+1)
            n = max(8, int(num_boundary * r / radius))
            a = np.linspace(0, 2*np.pi, n, endpoint=False) + np.pi/num_boundary*ring
            all_pts.append(np.c_[r*np.cos(a), r*np.sin(a)])
        all_pts.append([[0.0, 0.0]])
        pts_norm = np.vstack(all_pts)
        
        tri = Delaunay(pts_norm)
        valid_tris = []
        for simplex in tri.simplices:
            cent = np.mean(pts_norm[simplex], axis=0)
            if np.linalg.norm(cent) <= radius * 1.01:
                p0, p1, p2 = pts_norm[simplex]
                cross = (p1-p0)[0]*(p2-p0)[1] - (p1-p0)[1]*(p2-p0)[0]
                if abs(cross) >= 1e-8:
                    if cross < 0:
                        simplex = [simplex[0], simplex[2], simplex[1]]
                    valid_tris.append(simplex)
        
        pts = pts_norm * radius * boxsize / 2.0
        pts[:, 0] += cx
        pts[:, 1] += cy
        
        n = len(pts)
        model.particle_count = n
        model.particle_q = wp.array(pts.astype(np.float32), dtype=wp.vec2, device=device)
        model.particle_qd = wp.zeros(n, dtype=wp.vec2, device=device)
        model.particle_mass = wp.ones(n, dtype=float, device=device)
        model.particle_inv_mass = wp.ones(n, dtype=float, device=device)
        
        edges = set()
        for t in valid_tris:
            for e in [(t[0],t[1]), (t[1],t[2]), (t[2],t[0])]:
                edges.add(tuple(sorted(e)))
        spring_idx, spring_len = [], []
        for v0, v1 in edges:
            spring_idx.extend([v0, v1])
            spring_len.append(np.linalg.norm(pts[v1] - pts[v0]))
        
        model.spring_count = len(edges)
        model.spring_indices = wp.array(np.array(spring_idx, dtype=np.int32), dtype=int, device=device)
        model.spring_rest_length = wp.array(np.array(spring_len, dtype=np.float32), dtype=float, device=device)
        model.spring_stiffness = wp.full(model.spring_count, 40.0, dtype=float, device=device)
        model.spring_damping = wp.full(model.spring_count, 0.5, dtype=float, device=device)
        model.spring_strains = wp.zeros(model.spring_count, dtype=float, device=device)
        
        model.set_gravity((0.0, -0.1))
        print(f"Circle: {n} particles, {model.spring_count} springs")
        return model

print("State and Model defined")

State and Model defined


### 4.4 Explicit Solver

Uses **symplectic Euler** with springs only:
1. Evaluate forces at current position
2. Half-kick + drift + second half-kick

**Limitation:** Requires tiny timesteps ($\Delta t < 2\sqrt{m/k}$)

In [5]:
# Step 5: Explicit Solver
class SolverExplicit:
    """Explicit Euler solver - springs only, conditionally stable."""
    
    def __init__(self, model):
        self.model = model
    
    def step(self, state_in, state_out, dt):
        m = self.model
        state_in.particle_f.zero_()
        
        wp.launch(eval_spring_2d, dim=m.spring_count, inputs=[
            state_in.particle_q, state_in.particle_qd, m.spring_indices,
            m.spring_rest_length, m.spring_stiffness, m.spring_damping,
            state_in.particle_f, m.spring_strains], device=m.device)
        
        g = m.gravity.numpy()[0]
        grav = wp.vec2(g[0], g[1])
        
        wp.launch(integrate_particles_2d, dim=m.particle_count, inputs=[
            state_in.particle_q, state_in.particle_qd, state_in.particle_f,
            m.particle_inv_mass, grav, dt],
            outputs=[state_out.particle_q, state_out.particle_qd], device=m.device)
        
        wp.launch(apply_boundary_2d, dim=m.particle_count, inputs=[
            state_out.particle_q, state_out.particle_qd, m.boxsize], device=m.device)
        
        state_in.particle_f.zero_()
        wp.launch(eval_spring_2d, dim=m.spring_count, inputs=[
            state_out.particle_q, state_out.particle_qd, m.spring_indices,
            m.spring_rest_length, m.spring_stiffness, m.spring_damping,
            state_in.particle_f, m.spring_strains], device=m.device)
        
        wp.launch(finalize_velocity_2d, dim=m.particle_count, inputs=[
            state_out.particle_qd, state_in.particle_f, m.particle_inv_mass, grav, dt],
            outputs=[state_out.particle_qd], device=m.device)
        return state_out

print("SolverExplicit defined")

SolverExplicit defined


---
## 5. Simulation: Stability Comparison

We'll run **TWO simulations** with the same spring stiffness but different timesteps:
1. **STABLE**: Small timestep (dt = 0.001s) → Works fine
2. **UNSTABLE**: Large timestep (dt = 0.01s) → **EXPLODES!**

This demonstrates why explicit integration requires tiny timesteps.

In [ ]:
# Step 6: Run Simulation - captures frames even during explosion to visualize it
def run_simulation(model, solver, dt, sim_time, frame_interval, name="", explosion_threshold=10.0):
    """Run simulation, continue capturing frames even during explosion to visualize it."""
    steps = int(sim_time / dt)
    s_in, s_out = model.state(), model.state()
    print(f"Running {steps:,} {name} steps (dt={dt*1000:.2f}ms)...")
    start = time.time()
    frames = []
    exploded = False
    exploded_at_step = None
    
    for step in range(steps):
        solver.step(s_in, s_out, dt)
        s_in, s_out = s_out, s_in
        
        pos = s_in.particle_q.numpy()
        
        # Detect explosion (but don't stop - keep recording to show it!)
        if not exploded:
            max_displacement = np.max(np.abs(pos - np.array([center[0], center[1]])))
            if max_displacement > explosion_threshold or np.any(np.isnan(pos)):
                print(f"  ⚠️ EXPLOSION DETECTED at step {step} (t={step*dt:.3f}s)!")
                exploded = True
                exploded_at_step = step
        
        # Stop if positions become NaN or too extreme
        if np.any(np.isnan(pos)) or np.any(np.abs(pos) > 50):
            print(f"  💥 SIMULATION CRASHED at step {step}")
            break
            
        if step % frame_interval == 0:
            frames.append((
                pos.copy(),
                model.spring_strains.numpy().copy(),
                step * dt,
                exploded  # Track if this frame is during explosion
            ))
    
    elapsed = time.time() - start
    status = "EXPLODED" if exploded else "STABLE"
    print(f"  {name}: {len(frames)} frames captured, status: {status}")
    return frames, elapsed, exploded

# ============================================================
# SIMULATION PARAMETERS
# ============================================================
# Tutorial 1: Start with a SIMPLE, SMALL soft body
# - 12 boundary points, 2 inner rings → ~25 particles, ~60 springs
# - This is deliberately small to make explicit integration tractable
# Later tutorials will use LARGER bodies that require more advanced methods.
sim_time = 10.0
center = (1.5, 1)

# Spring stiffness - moderate value
# Stability limit: dt < 2*sqrt(m/k) = 2*sqrt(1/1000) ≈ 63ms
SPRING_STIFFNESS = 1000.0

# SAME body SIZE as all tutorials (1-3, 5): 20 boundary, 3 rings
# (FEM triangles added in Tutorial 3+, here we use springs only)
model_stable = Model.from_circle(radius=0.5, num_boundary=20, num_rings=3, boxsize=3.0, center=center)
model_stable.spring_stiffness = wp.full(model_stable.spring_count, SPRING_STIFFNESS, dtype=float, device=model_stable.device)

model_unstable = Model.from_circle(radius=0.5, num_boundary=20, num_rings=3, boxsize=3.0, center=center)
model_unstable.spring_stiffness = wp.full(model_unstable.spring_count, SPRING_STIFFNESS, dtype=float, device=model_unstable.device)

solver_stable = SolverExplicit(model_stable)
solver_unstable = SolverExplicit(model_unstable)

print(f"\n{'='*60}")
print(f"CONFIGURATION (identical for both simulations)")
print(f"{'='*60}")
print(f"  Particles:        {model_stable.particle_count}")
print(f"  Springs:          {model_stable.spring_count}")
print(f"  Spring stiffness: k = {SPRING_STIFFNESS:.0f}")
print(f"  Stability limit:  dt < {2*np.sqrt(1/SPRING_STIFFNESS)*1000:.1f}ms")
print(f"  Simulation time:  {sim_time}s")

# ============================================================
# RUN 1: STABLE (small timestep - below stability limit)
# ============================================================
dt_stable = 0.001  # 1ms - well below the 9ms stability limit
print(f"\n{'='*60}")
print(f"SIMULATION 1: STABLE (dt = {dt_stable*1000:.1f}ms < {2*np.sqrt(1/SPRING_STIFFNESS)*1000:.1f}ms limit)")
print(f"{'='*60}")
frames_stable, time_stable, exploded_stable = run_simulation(
    model_stable, solver_stable, dt_stable, sim_time, 
    frame_interval=50, name="Stable"
)

# ============================================================
# RUN 2: UNSTABLE (large timestep - causes explosion!)
# ============================================================
dt_unstable = 0.030  # 30ms - causes instability with k=1000
print(f"\n{'='*60}")
print(f"SIMULATION 2: UNSTABLE (dt = {dt_unstable*1000:.1f}ms > {2*np.sqrt(1/SPRING_STIFFNESS)*1000:.1f}ms limit)")
print(f"{'='*60}")
frames_unstable, time_unstable, exploded_unstable = run_simulation(
    model_unstable, solver_unstable, dt_unstable, sim_time,
    frame_interval=1, name="Unstable"  # Every frame to see explosion
)

print(f"\n{'='*60}")
print(f"RESULT")
print(f"{'='*60}")
print(f"  Stable   (dt={dt_stable*1000:.1f}ms < {2*np.sqrt(1/SPRING_STIFFNESS)*1000:.1f}ms): {'💥 EXPLODED' if exploded_stable else '✓ STABLE'}")
print(f"  Unstable (dt={dt_unstable*1000:.1f}ms > {2*np.sqrt(1/SPRING_STIFFNESS)*1000:.1f}ms): {'💥 EXPLODED' if exploded_unstable else '✓ STABLE'}")
print(f"\n>>> Same springs, same stiffness - only timestep differs!")
print(f">>> dt above stability limit → EXPLOSION!")

TypeError: Model.from_circle() got an unexpected keyword argument 'use_fem'

---
## 6. Visualization: Side-by-Side Comparison

Watch the **STABLE** simulation (small timestep) vs the **UNSTABLE** simulation (large timestep) that explodes!

In [ ]:
# Step 7: Side-by-Side Animation - STABLE vs UNSTABLE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor('white')
plt.rcParams['text.color'] = 'black'

spring_idx = model_stable.spring_indices.numpy().reshape(-1, 2)
cmap_spring = LinearSegmentedColormap.from_list('spring', ['#FFE066', '#FF8800', '#CC0000'])
collision_thresh = 0.15
boxsize = model_stable.boxsize

def get_colors(pos, boxsize):
    """Pink near boundary, light blue otherwise (same as original)."""
    return ['#FF69B4' if (p[0]<collision_thresh or p[0]>boxsize-collision_thresh or 
                          p[1]<collision_thresh or p[1]>boxsize-collision_thresh) 
            else '#87CEEB' for p in pos]

# Use maximum frames to show the full stable simulation
n_frames = max(len(frames_stable), len(frames_unstable) + 30)
print(f"Animating {n_frames} frames (stable: {len(frames_stable)}, unstable: {len(frames_unstable)})")

def update(idx):
    # LEFT: STABLE simulation
    if idx < len(frames_stable):
        pos, strain, t, is_exploding = frames_stable[idx]
        ax1.clear()
        ax1.set_facecolor('white')
        
        # Draw springs colored by strain
        segs = [[pos[v0], pos[v1]] for v0, v1 in spring_idx]
        lc = LineCollection(segs, cmap=cmap_spring, linewidths=2.5)
        lc.set_array(np.clip(np.abs(strain)/max(0.02, np.abs(strain).max()), 0, 1))
        ax1.add_collection(lc)
        
        # Draw particles (pink when near boundary)
        colors = get_colors(pos, boxsize)
        ax1.scatter(pos[:,0], pos[:,1], s=35, c=colors, edgecolors='white', linewidths=0.8, zorder=5)
        
        # SAME view bounds as unstable panel
        ax1.set_xlim(0, boxsize); ax1.set_ylim(0, boxsize)
        ax1.set_aspect('equal'); ax1.set_xticks([]); ax1.set_yticks([])
        ax1.set_title(f"STABLE (dt={dt_stable*1000:.1f}ms) | t={t:.2f}s", fontweight='bold', color='black')
    
    # RIGHT: UNSTABLE simulation
    if idx < len(frames_unstable):
        pos, strain, t, is_exploding = frames_unstable[idx]
        ax2.clear()
        ax2.set_facecolor('white')
        
        # Draw springs colored by strain (no clipping - show actual positions)
        segs = [[pos[v0], pos[v1]] for v0, v1 in spring_idx]
        lc = LineCollection(segs, cmap=cmap_spring, linewidths=2.5)
        strain_val = np.abs(strain)
        lc.set_array(np.clip(strain_val/max(0.02, strain_val.max()+1e-6), 0, 1))
        ax2.add_collection(lc)
        
        # Draw particles
        colors = get_colors(pos, boxsize)
        ax2.scatter(pos[:,0], pos[:,1], s=35, c=colors, edgecolors='white', linewidths=0.8, zorder=5)
        
        # SAME view bounds as stable panel - explosion will go outside view
        ax2.set_xlim(0, boxsize); ax2.set_ylim(0, boxsize)
        ax2.set_aspect('equal'); ax2.set_xticks([]); ax2.set_yticks([])
        status = " - EXPLODING!" if is_exploding else ""
        ax2.set_title(f"UNSTABLE (dt={dt_unstable*1000:.1f}ms) | t={t:.2f}s{status}", fontweight='bold', color='black')
    else:
        # Show "crashed" message after simulation ends
        ax2.clear()
        ax2.set_facecolor('white')
        ax2.text(0.5, 0.5, "SIMULATION\nCRASHED!", ha='center', va='center', 
                fontsize=24, fontweight='bold', color='black', transform=ax2.transAxes)
        ax2.set_xlim(0, boxsize); ax2.set_ylim(0, boxsize)
        ax2.set_aspect('equal'); ax2.set_xticks([]); ax2.set_yticks([])
        ax2.set_title(f"UNSTABLE (dt={dt_unstable*1000:.1f}ms) | CRASHED", fontweight='bold', color='black')
    
    plt.tight_layout()

anim = FuncAnimation(fig, update, frames=n_frames, interval=40, blit=False)
anim.save('explicit_stability_comparison.gif', writer=PillowWriter(fps=25))
plt.close()
print(f"Saved explicit_stability_comparison.gif")

In [ ]:
from IPython.display import Image as IPImage
IPImage(filename='explicit_stability_comparison.gif')

---
## 7. Results & Discussion

In [ ]:
# Step 8: Summary Statistics
print("=" * 60)
print("         EXPLICIT EULER STABILITY DEMONSTRATION")
print("=" * 60)
print(f"\n{'Configuration (IDENTICAL for both):'}")
print(f"  Particles:        {model_stable.particle_count}")
print(f"  Springs:          {model_stable.spring_count}")
print(f"  Spring stiffness: k = {SPRING_STIFFNESS:.0f}")
print(f"  Stability limit:  dt < {2*np.sqrt(1/SPRING_STIFFNESS)*1000:.1f}ms")
print(f"\n{'Results:'}")
print(f"  ✓ STABLE   (dt = {dt_stable*1000:.1f}ms < limit): Completed {sim_time}s simulation")
print(f"  ✗ UNSTABLE (dt = {dt_unstable*1000:.1f}ms > limit): {'💥 EXPLODED!' if exploded_unstable else 'Completed'}")
print(f"\n{'Why does explicit explode when dt > limit?'}")
print(f"  At each step, explicit overshoots:")
print(f"    → Springs compress/extend too far")
print(f"    → Force grows larger than it should")
print(f"    → Next step overshoots even more")
print(f"    → Positive feedback → EXPLOSION!")
print("=" * 60)
print(f"\n>>> The SAME stiff springs + large dt work with IMPLICIT integration!")
print(f">>> See 02_implicit.ipynb for the comparison.")

---
## 8. Summary

**What we demonstrated:**
- Spring force kernel using Hooke's Law
- Symplectic Euler integration on GPU  
- ⚠️ **Explicit integration EXPLODES with large timesteps!**

**Key Takeaways:**
- **Explicit integration** is simple but **conditionally stable**
- With spring stiffness k=1,000 and dt=30ms → **explosion**
- The same timestep and stiffness work perfectly with implicit integration!

**The Problem (same body, same springs, same stiffness):**
| Method | dt=1ms | dt=30ms |
|--------|--------|---------|
| Explicit | ✓ Works | 💥 Explodes |
| Implicit | ✓ Works | ✓ Works |

**Next:** In [02_implicit.ipynb](02_implicit.ipynb), we'll show that **implicit integration handles the SAME parameters with no problems!**

✅ **Tutorial complete!** Continue to [02_implicit.ipynb](02_implicit.ipynb) to see how implicit integration solves this problem.